<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

Pages are resized and boxes are read off the detector map the same way as PaddleOCR: keep the aspect ratio, then take DB contours, a minimum-area rectangle, and an unclip. Recognition is CTC over the LightSVTR sequence. Compilation stays inside [`main`](https://alix559.github.io/solid_lens/download.html#main), so the geometry below can be read on its own.

## Detector size

These are the PP-OCRv6 medium overrides from PaddleOCR's OCR pipeline. The checkpoint yaml is looser (0.2 / 0.45 / 1.4). The pipeline that produces the boxes uses the numbers below.

In [6]:
#| echo: false
#| output: asis
show_doc(detection_hw)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L40){target="_blank" style="float:right; font-size:smaller"}

### detection_hw

```python
def detection_hw(
    height:int, # Page height, in pixels
    width:int, # Page width, in pixels
    limit_side_len:int=64, # Side the limit type compares against
    limit_type:str='min', # `min`, `max`, or `resize_long`
    max_side_limit:int=4000
)->tuple[int, int]:
```

*Detector height and width. Same rule as PaddleOCR `DetResizeForTest` type 0.*

`limit_type` is `min` and `limit_side_len` is 64, so a page whose short side is already at least 64 is not shrunk. Both sides are then rounded to a multiple of 32, and nothing is allowed below 32. `round` is bankers' rounding, so a half step lands on the even multiple.

In [7]:
from fastcore.test import test_eq

test_eq(detection_hw(100, 200), (96, 192))
test_eq(detection_hw(40, 80), (64, 128))
test_eq(detection_hw(100, 400, limit_side_len=64, limit_type="resize_long"), (32, 64))
detection_hw(100, 200)

(96, 192)

## CTC

Class 0 is the blank. Repeated labels collapse, and blanks separate the repeats that should survive.

In [9]:
from fastcore.test import test_eq

test_eq(_ctc_decode([0, 1, 1, 0, 2, 2, 3], ["a", "b", "c"]), "abc")
test_eq(_ctc_decode([1, 1], ["a"]), "a")
_ctc_decode([0, 1, 0, 1], ["a", "b"])

'aa'

## Images

Detector normalization is ImageNet mean and std on a BGR image. The recognizer uses a different scale, `(x / 255 - 0.5) / 0.5`, applied later per crop.

In [13]:
#| echo: false
#| output: asis
show_doc(load_charset)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L91){target="_blank" style="float:right; font-size:smaller"}

### load_charset

```python
def load_charset(
    path:Path, *, add_space:bool=False
)->list[str]:
```

In [15]:
#| echo: false
#| output: asis
show_doc(load_image)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L99){target="_blank" style="float:right; font-size:smaller"}

### load_image

```python
def load_image(
    path:Path
)->np.ndarray:
```

*Return an RGB uint8 image.*

In [17]:
#| echo: false
#| output: asis
show_doc(resize_hwc)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L106){target="_blank" style="float:right; font-size:smaller"}

### resize_hwc

```python
def resize_hwc(
    image:np.ndarray, out_h:int, out_w:int
)->np.ndarray:
```

This is bilinear resampling with the pixel center at the half coordinate, which is what the recognizer crops use. An image that is already the requested size is returned unchanged.

In [18]:
import numpy as np
from fastcore.test import test_eq

image = np.arange(12, dtype=np.uint8).reshape(3, 4, 1).repeat(3, axis=2)
same = resize_hwc(image, 3, 4)
test_eq(same.shape, image.shape)
test_eq(bool(same is image), True)
small = resize_hwc(image, 1, 1)
test_eq(small.shape, (1, 1, 3))
small

array([[[5, 5, 5]]], dtype=uint8)

In [20]:
#| echo: false
#| output: asis
show_doc(det_input)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L126){target="_blank" style="float:right; font-size:smaller"}

### det_input

```python
def det_input(
    image_rgb:np.ndarray, out_h:int, out_w:int
)->np.ndarray:
```

*Resize like Paddle and normalize to a BGR NCHW tensor.*

In [21]:
import numpy as np
from fastcore.test import test_eq

page = np.zeros((8, 16, 3), np.uint8)
tensor = det_input(page, 32, 32)
test_eq(tensor.shape, (1, 3, 32, 32))
test_eq(tensor.dtype, np.float32)
tensor.shape

(1, 3, 32, 32)

## Boxes

The unclip distance is the DB post-process offset: area times the ratio, divided by the perimeter. Round joins match PaddleOCR.

In [26]:
#| echo: false
#| output: asis
show_doc(boxes_from_bitmap)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L180){target="_blank" style="float:right; font-size:smaller"}

### boxes_from_bitmap

```python
def boxes_from_bitmap(
    prob:np.ndarray, dest_width:int, dest_height:int, *, thresh:float=0.3, box_thresh:float=0.6,
    unclip_ratio:float=1.5, max_candidates:int=3000
)->list[tuple[int, int, int, int, float]]:
```

*PaddleOCR `DBPostProcess` quads, returned as axis-aligned boxes.*

Coordinates come back in the original image. The recognizer crops that rectangle. Boxes are sorted top to bottom, then left to right. A contour whose short side is under 3 pixels, or whose mean score is under 0.6, is dropped. After the unclip the short side must clear 5.

In [27]:
import numpy as np
from fastcore.test import test_eq

prob = np.zeros((64, 64), np.float32)
prob[20:36, 8:48] = 0.95
boxes = boxes_from_bitmap(prob, 64, 64)
test_eq(len(boxes), 1)
left, top, right, bottom, score = boxes[0]
test_eq(score > 0.6, True)
test_eq(left < 8, True)
test_eq(right > 48, True)
boxes[0]

(0, 12, 55, 43, 0.949999988079071)

## Recognition

In [30]:
#| echo: false
#| output: asis
show_doc(recognize_crop)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L245){target="_blank" style="float:right; font-size:smaller"}

### recognize_crop

```python
def recognize_crop(
    model, image_rgb:np.ndarray, charset:list[str]
)->str:
```

A line that fits in 320 columns is one forward pass. A longer line is a sliding window. Five timesteps are dropped on each interior edge, where the crop has no context, and the stride is chosen so the kept timesteps abut. The network's width stride is 8.

In [32]:
#| echo: false
#| output: asis
show_doc(detect_boxes)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L284){target="_blank" style="float:right; font-size:smaller"}

### detect_boxes

```python
def detect_boxes(
    image_rgb:np.ndarray, det_model
)->list[tuple[int, int, int, int, float]]:
```

*Text boxes in original pixels, from one forward pass at the PaddleOCR det size.*

In [34]:
#| echo: false
#| output: asis
show_doc(read_page)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L293){target="_blank" style="float:right; font-size:smaller"}

### read_page

```python
def read_page(
    image_rgb:np.ndarray, det_model, rec_model, charset:list[str]
)->list[tuple[str, float, int, int, int, int]]:
```

[`read_page`](https://alix559.github.io/solid_lens/ocr.html#read_page) drops crops the recognizer turns into an empty string. The score on each line is the detector score of that box, not a recognition probability.

In [36]:
#| echo: false
#| output: asis
show_doc(draw_boxes)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L308){target="_blank" style="float:right; font-size:smaller"}

### draw_boxes

```python
def draw_boxes(
    image_rgb:np.ndarray, lines:list[tuple[str, float, int, int, int, int]]
)->np.ndarray:
```

*Paint a green rectangle around each recognized line.*

In [38]:
#| echo: false
#| output: asis
show_doc(save_rgb)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/ocr.py#L327){target="_blank" style="float:right; font-size:smaller"}

### save_rgb

```python
def save_rgb(
    path:Path, image_rgb:np.ndarray
)->None:
```

## Command

In [40]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/download.py#L52){target="_blank" style="float:right; font-size:smaller"}

### main

```python
def main()->None:
```